# Variante `multi_heatmap` — Universal EEG Transformer


El **Universal EEG Transformer** en modo **multi-configuración** entrena **un
solo modelo** que acepta grabaciones en *cualquier* configuración de
electrodos y predice, **en esa misma configuración**, las medidas con una
referencia distinta (conversión de referencia intra-configuración).

### Las configuraciones (`mapping.configs`)

| Config | Canales | Origen |
|---|---|---|
| `10-20` | 19 | Subconjunto exacto de los 64 canales (selección de columnas) |
| `10-10` | 39 | Subconjunto exacto de los 64 canales |
| `canonical` | 64 | El montaje completo del dataset |
| `dense-128` | 128 | **Fundido denso**: mismo campo escalar REST interpolado |
| `dense-256` | 256 | **Fundido denso**: mismo campo escalar REST interpolado |

### Cómo se construyen las observaciones (fiel a la física)

1. **Ancla única**: la referencia al infinito (REST) del montaje canónico,
   obtenida con el lead field multicapa analítico (`T_canon pg`).
2. Cada configuración observa ese campo en sus propias posiciones:
   los subconjuntos reales son **selección de columnas** del ancla; los densos
   son **interpolación esférica (Perrin)** del ancla a `128/256` posiciones
   en el casquete (fibonacci, radio = mediana de la norma canónica).
3. Sobre esa observación se computan las 4 referencias **del propio montaje**
   (con su lead field): unipolar/bipolar/CAR locales y REST de configuración.
4. Proyecciones fijas `P_s`/`Q_s`: para subconjuntos, selección exacta
   (`Q` selecciona las columnas del core canonico); para densos, splines. La
   ruta efectiva `s→s` es `A_s = Q_s·core·P_s`.

### Fairness del experimento

- **Balanceo por configuración**: todas aportan las mismas muestras por
  lote/época (`multi_max_samples_per_split` por split), sin sesgar la
  minimización hacia las configs más densas.
- **Inicialización lineal empírica** sobre el canónico (`latent_dim = C`):
  el latente ancla al unipolar canónico y cada ruta arranca bien condicionada.
- **Pérdida**: MSE estandarizado por ruta (Z-score por lote) + MSE real.
- **Evaluación (test)**: RMSE/correlación/variación explicada **por
  configuración** (diagonal y cruzada entre referencias) comparadas con la
  **línea base analítica** `T_d @ pinv(T_s)` del propio montaje (columnas
  `*_ana`): la ganancia del modelo se lee restando ambas.

### Notas

- Las configs `dense-N` son simuladas (interpolación, no registros reales);
  las 10-20/10-10 son subconjuntos reales de los 64 canales (PhysioNet eegbci).
- Conceptos completos (física lead-field/REST, métricas, lectura de
  resultados): `docs/guia_conceptual.md`.

### Multi_heatmap: el "heatmap" es una salida entrenada

La variante **`multi_heatmap`** añade a `multi_montage` la lectura de la
actividad como **campo de superficie sobre una malla compartida** del cuero
cabelludo. Para cada configuración la observación se interpola a los
`grid_px²` nodos con la matriz fija `S_s (C_s → n_grid)` (la misma
`scalp_grid_matrix` de los topomapas):

$$\hat F^{(d)}_s = \hat x^{(d)}_s\, S_s^\top$$

El término de pérdida es el MSE estandarizado **sobre el patrón espacial**
(peso `model.surface_loss_weight = 0.1`), de modo que el modelo no solo acierta
electrodo a electrodo sino también la forma de las "manchas". Al ser `S_s`
fijo, todos los campos viven en la **misma malla**: los heatmaps de 19/64/128/
256 canales son comparables entre sí.


> **Nota de reproducción:** el modelo ya entrenado (12 sujetos) está en `runs/multi_heatmap`. Con `FORCE = False` el notebook **reutiliza el checkpoint** sin reentrenar (segundos); póngalo en `True` solo para reentrenar desde cero.

## 1. Carga del experimento

Configuración YAML, dataset real cacheado y los **insumos multi-configuración** (observaciones y referencias por configuración).

In [ ]:
# ---- Configuración del entorno ----------------------------------------
# Renderizado inline de figuras (debe activarse antes de importar matplotlib)
%matplotlib inline

import os, sys
from pathlib import Path

# Ruta raíz del repo y paquetes propios (src/)
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eeg_transform.experiments.multi import multiconfig_summary
from eeg_transform.nb import (
    config_table, evaluate_multiconfig, evaluate_multiconfig_surface,
    load_experiment, multiconfig_data,
    plot_multiconfig_bars, plot_multiconfig_heatmap, plot_multiconfig_scalps,
    plot_multiconfig_surface, plot_training, train_variant,
)
from eeg_transform.training.trainer import (
    build_multiconfig_model, is_multiconfig_variant,
)

np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)
plt.rcParams["figure.dpi"] = 110

CONFIG = ROOT / "config/multi_heatmap.yaml"
FORCE  = False          # True = reentrenar desde cero ignorando el checkpoint

In [ ]:
cfg, ds = load_experiment(CONFIG)
data = multiconfig_data(cfg, ds)
assert is_multiconfig_variant(cfg), f"Variant {cfg.model.variant} no es multi_montage/multi_heatmap"
print(ds.summary())
print(multiconfig_summary(data), "\n")
config_table(cfg).set_index(["sección", "parámetro"])

## 2. Arquitectura

Un autoencoder universal en el canónico + **proyecciones fijas** `P_s`/`Q_s` por configuración. Las matrices efectivas `A_s→s = Q_s·core·P_s` convierten las 4 referencias dentro de cada configuración.

In [ ]:
# Un latente canónico + proyecciones fijas por configuración P_s/Q_s
model = build_multiconfig_model(cfg, data)

n_params = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
print(f"Variante: {cfg.model.variant}  |  latente canónico: {model.n_canonical}  |  "
      f"configs: {len(model.configs)}  |  parámetros: {n_params:,}")
for lbl in model.configs:
    A = model.transfer_matrices(lbl)
    m = next(iter(A.values()))
    print(f"  {lbl:10s} P {model.projections[lbl].shape}  "
          f"Q {model.out_maps[lbl].shape}  A_s→d {m.shape}")
print("La predicción es INTRA-configuración: C_s → C_s (misma disposición).")
_ = model  # se reutiliza en train/eval

## 3. Entrenamiento

Adam (lr 1e-3), Z-score por lote, early stopping y reduce-LR. Balanceado: mismas muestras por config/lote. Reutiliza el checkpoint si existe.

In [ ]:
# Entrenamiento balanceado (mismas muestras por config/época); reutiliza el
# checkpoint de runs/{VARIANT}/best.weights.h5 si existe (salvo FORCE=True).
model, history = train_variant(cfg, ds, force=FORCE)
run_dir = Path(cfg.training.run_dir)
print(f"run_dir: {run_dir}")

## 4. Evaluación en test

Métricas por configuración (diagonal/cruzada entre referencias) frente a la **línea base analítica** `T_d @ pinv(T_s)` del propio montaje (columnas `*_ana`).

In [ ]:
# Evaluación en test: RMSE/ve por configuración, diagonal y cruzada; las
# columnas *_ana son la línea base analítica T_d @ pinv(T_s) sobre el ancla.
metrics_df, summary = evaluate_multiconfig(cfg, ds, model, data)

print("=== RESUMEN MULTI-CONFIG (RMSE en µV): modelo vs línea base analítica ===")
print(summary.round(3).to_string(index=False))
print("\n=== DETALLE POR CONFIGURACIÓN (test) ===")
print(metrics_df.round(9).to_string(index=False))

## 5. Figuras inline

Curvas de aprendizaje, heatmap de RMSE por config origen→destino, barras modelo vs análisis, y los **mapas de calor del cuero cabelludo** por configuración (observación → modelo → verdad).

In [ ]:
# Curvas de aprendizaje (pérdida estandarizada y MSE real)
plot_training(run_dir / "history.csv")
plt.show()

In [ ]:
# Heatmap de RMSE real por config origen→destino (µV, escala log10)
plot_multiconfig_heatmap(metrics_df,
                         title=f"RMSE real multi-config (µV, log10) — {cfg.model.variant}")
plt.show()

In [ ]:
# Barras: RMSE y ve por configuración, diagonal y cruzada, frente a la
# línea base analítica (*_ana) del propio montaje.
plot_multiconfig_bars(metrics_df,
                      title=f"RMSE/ve multi-config vs análisis — {cfg.model.variant}")
plt.show()

In [ ]:
# Mapas de calor del cuero cabelludo por configuración. Filas = referencia
# (unipolar local, bipolar local, CAR, REST); columnas: observación →
# modelo (predicción intra-config) → verdad. El ancla REST es la referencia
# infinita simulada y las demás se derivan de sus operadores.
for label, fig in plot_multiconfig_scalps(cfg, model, data=data).items():
    print(f"--- {label} ---")
    plt.show()

## 6. Campo de superficie (heatmap

La lectura como campo en la malla compartida es una salida **entrenada** (peso `surface_loss_weight`): aquí se mide RMSE/r/VE del patrón espacial y se compara por configuración.

In [ ]:
# Campo de superficie (solo ``multi_heatmap``): RMSE/r/VE del HEATMAP.
# La lectura se interpola con la matriz fija S_s (electrodos -> malla
# compartida); el término de pérdida del entrenamiento actúa sobre estas
# "manchas", no solo sobre cada electrodo por separado.
surface_df, surf_summary = evaluate_multiconfig_surface(model, data)

print("=== CAMPO DE SUPERFICIE (TEST) — RMSE en µV sobre la malla ===")
print(surf_summary.round(3).to_string(index=False))
print("\n=== DETALLE POR RUTA (campo, malla compartida) ===")
print(surface_df.round(9).to_string(index=False))

In [ ]:
# Barras del campo de superficie por configuración: RMSE en la malla y VE
# del heatmap predicho (diagonal y cruzada entre referencias).
plot_multiconfig_surface(surface_df,
                         title=f"Campo de superficie (heatmap) — {cfg.model.variant}")
plt.show()

## Conclusiones

Consulte `docs/guia_conceptual.md` (conceptos y métricas), `docs/results_comparison.md` (comparativa de variantes) y los resultados guardados en `runs/multi_heatmap/metrics_test.csv` (y `runs/multi_heatmap/metrics_surface_test.csv` para `multi_heatmap`).